# 🍕 Linear Regression Deep Dive with Pizza Store Data

**Complete Linear Regression Course** covering:
- Simple Linear Regression (by hand + sklearn)
- Multiple Linear Regression
- Normal Equation vs Gradient Descent
- Assumptions & Diagnostics
- Evaluation Metrics (MSE, RMSE, R², Adjusted R²)
- Regularization (Ridge, Lasso, Elastic Net)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)

### 🤔 Understanding the dataset

We create 200 pizza stores with a **known true relationship**:

`daily_sales = 30 + 25×employees + 50×rating + 1.5×ad_spend + noise`

The `noise` simulates real-world randomness — even with the same employees, rating, and ad spend, 
two stores won't have identical sales. By knowing the true formula, we can check if our model 
recovers the correct coefficients.

In [ ]:
# Create Pizza Store Dataset
n_stores = 200
np.random.seed(42)

employees = np.random.randint(2, 12, n_stores)
rating = np.round(np.random.uniform(2.5, 5.0, n_stores), 1)
ad_spend = np.random.randint(50, 300, n_stores)

# True relationship: sales = 30 + 25*employees + 50*rating + 1.5*ad_spend + noise
daily_sales = (30 + 25*employees + 50*rating + 1.5*ad_spend
               + np.random.normal(0, 30, n_stores)).astype(int)

df = pd.DataFrame({
    'employees': employees,
    'rating': rating,
    'ad_spend': ad_spend,
    'daily_sales': daily_sales
})

print(f'Dataset: {len(df)} pizza stores')
print(f'Sales range: ${df.daily_sales.min()} — ${df.daily_sales.max()}')
df.head(8)

---
## Part 1: Simple Linear Regression — By Hand

### 1.1 The Math: β₁ = Cov(x,y) / Var(x), β₀ = ȳ − β₁x̄

### 🤔 What are Covariance and Variance, and why do we need them?

To find the best-fit line, we need two statistics:

---

**Covariance (Cov)** measures how two variables move **together**:
- Positive Cov: when x goes up, y tends to go up (more employees → more sales)
- Negative Cov: when x goes up, y tends to go down
- Zero Cov: no relationship

**Formula:**
```
Cov(x, y) = (1/n) × Σ (xᵢ − x̄)(yᵢ − ȳ)
```

**How to read this:** For each data point, ask: "How far is this x from the average x?" and "How far is this y from the average y?" Multiply those two distances together. Then average across all data points.

**Step-by-step example** (with 4 stores):
```
Store  Employees(x)  Sales(y)   (xᵢ − x̄)   (yᵢ − ȳ)   Product
─────  ────────────  ────────   ─────────   ─────────   ───────
  S1        5          520       5−3.5=1.5   520−400=120   1.5 × 120 = 180
  S2        2          280       2−3.5=−1.5  280−400=−120  −1.5 × −120 = 180
  S3        4          450       4−3.5=0.5   450−400=50    0.5 × 50 = 25
  S4        3          350       3−3.5=−0.5  350−400=−50   −0.5 × −50 = 25

x̄ = (5+2+4+3)/4 = 3.5       ȳ = (520+280+450+350)/4 = 400

Cov(x,y) = (180 + 180 + 25 + 25) / 4 = 410 / 4 = 102.5
```

**Key insight:** When x is above average AND y is above average, the product is **positive**. When both are below average, the product is also **positive** (negative × negative). So if x and y move together, the covariance is positive.

---

**Variance (Var)** measures how spread out a single variable is:
- High Var: values are far from the mean
- Low Var: values are clustered near the mean

**Formula:**
```
Var(x) = (1/n) × Σ (xᵢ − x̄)²
```

**How to read this:** For each data point, ask: "How far is this x from the average?" Square that distance (to make it positive). Then average across all data points.

**Step-by-step example** (same 4 stores):
```
Store  Employees(x)   (xᵢ − x̄)   (xᵢ − x̄)²
─────  ────────────   ─────────   ──────────
  S1        5          1.5         2.25
  S2        2         −1.5         2.25
  S3        4          0.5         0.25
  S4        3         −0.5         0.25

Var(x) = (2.25 + 2.25 + 0.25 + 0.25) / 4 = 5.0 / 4 = 1.25
```

Notice: Variance is just Covariance of x with **itself** — Var(x) = Cov(x, x).

---

**Why β₁ = Cov(x,y) / Var(x)?**

```
β₁ = Cov(x,y) / Var(x) = 102.5 / 1.25 = 82.0
```

The slope tells us: "for each unit increase in x, how much does y change?" Covariance captures the joint movement, and dividing by Var(x) **normalizes** it to a per-unit-of-x scale.

Think of it this way:
- Cov(x,y) tells you the **raw** relationship strength
- Var(x) tells you how much x varies
- Dividing gives you the relationship **per unit of x** — which is exactly what a slope is!

This is the **Ordinary Least Squares (OLS)** formula — it gives the line that minimizes the sum of squared errors.

In [ ]:
x = df['employees'].values
y = df['daily_sales'].values

# Step-by-step calculation
x_mean = x.mean()
y_mean = y.mean()
cov_xy = np.mean((x - x_mean) * (y - y_mean))
var_x = np.mean((x - x_mean) ** 2)

beta_1 = cov_xy / var_x
beta_0 = y_mean - beta_1 * x_mean

print('SIMPLE LINEAR REGRESSION — BY HAND')
print('=' * 50)
print(f'x̄ (mean employees) = {x_mean:.2f}')
print(f'ȳ (mean sales)     = ${y_mean:.2f}')
print(f'Cov(x, y)          = {cov_xy:.2f}')
print(f'Var(x)             = {var_x:.2f}')
print(f'\nβ₁ = Cov/Var = {cov_xy:.2f} / {var_x:.2f} = {beta_1:.2f}')
print(f'β₀ = ȳ − β₁x̄ = {y_mean:.2f} − {beta_1:.2f}×{x_mean:.2f} = {beta_0:.2f}')
print(f'\n📐 Model: ŷ = {beta_0:.2f} + {beta_1:.2f} × Employees')
print(f'\nInterpretation:')
print(f'  Each additional employee → +${beta_1:.2f} daily sales')
print(f'  Baseline (0 employees)  → ${beta_0:.2f} (theoretical)')

### 1.2 Visualize the Best-Fit Line

### 🤔 Why visualize the regression line?

A plot of the data with the best-fit line helps you:
- **Verify** the model makes sense (does the line follow the data?)
- **Spot problems** like outliers pulling the line, or a curved pattern that a straight line misses
- **Communicate** results to non-technical stakeholders

The green dashed lines are **residuals** — the vertical distance from each point to the line. 
The model minimizes the sum of these squared distances.

In [ ]:
y_pred_simple = beta_0 + beta_1 * x

plt.figure(figsize=(10, 6))
plt.scatter(x, y, alpha=0.5, color='steelblue', label='Actual stores')
plt.plot(sorted(x), [beta_0 + beta_1*xi for xi in sorted(x)],
         'r-', linewidth=2, label=f'ŷ = {beta_0:.1f} + {beta_1:.1f}x')

# Draw residuals for a few points
for i in range(0, 15, 3):
    plt.plot([x[i], x[i]], [y[i], y_pred_simple[i]],
             'g--', alpha=0.5, linewidth=1)

plt.xlabel('Employees')
plt.ylabel('Daily Sales ($)')
plt.title('Simple Linear Regression: Sales vs Employees')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 1.3 Verify with sklearn

In [ ]:
model_simple = LinearRegression()
model_simple.fit(x.reshape(-1, 1), y)

print('sklearn vs Hand Calculation:')
print(f'  β₀: sklearn={model_simple.intercept_:.2f}, hand={beta_0:.2f}')
print(f'  β₁: sklearn={model_simple.coef_[0]:.2f}, hand={beta_1:.2f}')
print('\n✅ They match!')

---
## Part 2: Evaluation Metrics

### 🤔 Why do we need evaluation metrics?

A model is useless if we can't measure how good it is. Key metrics:

- **MSE** (Mean Squared Error): Average of squared errors. Penalizes large errors heavily.
- **RMSE** (Root MSE): Square root of MSE — same units as y (dollars). Easier to interpret.
- **MAE** (Mean Absolute Error): Average of absolute errors. Less sensitive to outliers.
- **R²** (R-squared): Proportion of variance explained. 0 = terrible, 1 = perfect.
- **Adjusted R²**: R² penalized for extra features — prevents gaming R² by adding useless features.

---

### �� Why Mean SQUARED Error? Why not just Mean Error?

This is a great question. Let's see what happens with plain Mean Error:

```
Store   Actual   Predicted   Error (actual − predicted)
─────   ──────   ─────────   ─────────────────────────
  S1     $520      $530        520 − 530 = −10  (overpredicted)
  S2     $310      $290        310 − 290 = +20  (underpredicted)
  S3     $450      $460        450 − 460 = −10  (overpredicted)

Mean Error = (−10 + 20 + −10) / 3 = 0 / 3 = 0  ← LOOKS PERFECT!
```

**The problem:** Positive and negative errors **cancel each other out**. A Mean Error of 0 doesn't mean the model is perfect — it just means the overestimates and underestimates happen to balance. The model is still off by $10–$20 on every prediction!

**Three ways to fix this:**

```
Option 1: Mean Absolute Error (MAE) — take |absolute value| of each error
  MAE = (|−10| + |20| + |−10|) / 3 = (10 + 20 + 10) / 3 = 13.33

Option 2: Mean Squared Error (MSE) — square each error
  MSE = ((-10)² + 20² + (-10)²) / 3 = (100 + 400 + 100) / 3 = 200

Option 3: RMSE — take √MSE to get back to dollar units
  RMSE = √200 = $14.14
```

**Both MAE and MSE solve the cancellation problem. So why prefer MSE?**

| Property | MAE | MSE |
|----------|-----|-----|
| Cancellation problem | ✅ Solved | ✅ Solved |
| Penalizes large errors | Equally (linear) | **More heavily** (quadratic) |
| Differentiable everywhere | ❌ Has a kink at 0 | ✅ Smooth everywhere |
| Optimization | Harder | **Easier** (gradient descent works better) |
| Sensitive to outliers | Less | More |

**The key reasons MSE is the default:**

1. **Squaring punishes big errors much more.** An error of 10 costs 100, but an error of 20 costs 400 (4× more, not 2×). This is often what we want — a model that's off by $100 once is worse than one that's off by $10 ten times.

2. **MSE is smooth (differentiable everywhere).** MAE has a sharp corner (kink) at error = 0, which makes gradient descent struggle. MSE's smooth parabolic shape gives clean gradients that point straight to the minimum.

```
  Loss                          Loss
   |\      /|  MAE               |         MSE
   | \    / |  (V-shape,         |  \     /
   |  \  /  |   kink at 0)       |   \   /
   |   \/   |                    |    \_/   (smooth bowl,
   +--------→ error              +--------→  easy to optimize)
```

3. **MSE has a closed-form solution.** Because of the smooth math, we can derive the Normal Equation (β = (XᵀX)⁻¹Xᵀy) directly. With MAE, there's no closed-form — you can only use iterative methods.

**When to use MAE instead:** When you have outliers and don't want them to dominate the loss. For example, if one store has a data entry error ($50,000 instead of $500), MSE would be dominated by that single point, while MAE would handle it more gracefully.

**Bottom line:** MSE is the default because it's mathematically convenient and penalizes big errors. Use MAE when outliers are a concern.

### 📊 MAE vs MSE — See the Difference with Real Numbers

Let's use example pizza store data to see how MAE and MSE behave differently — especially how they respond to errors of different sizes and how their loss curves look.

**Quick refresher — what is a gradient?**

The gradient answers: *"Which direction should I move my weights to reduce error, and by how much?"*

It has two parts:
- **Direction**: move left or right? (sign of the gradient)
- **Magnitude**: take a big step or a tiny step? (size of the gradient)

This magnitude part is where MAE and MSE differ dramatically. Watch for it in the plots below.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- Example 1: How MAE and MSE react to individual errors ---
stores = ['Store A', 'Store B', 'Store C', 'Store D', 'Store E']
actual =    np.array([500, 320, 450, 600, 280])
predicted = np.array([510, 290, 460, 520, 300])
errors = actual - predicted

comparison = pd.DataFrame({
    'Store': stores,
    'Actual ($)': actual,
    'Predicted ($)': predicted,
    'Error': errors,
    '|Error| (MAE)': np.abs(errors),
    'Error² (MSE)': errors**2
})

print('MAE vs MSE — Per-Store Breakdown')
print('=' * 70)
print(comparison.to_string(index=False))
print(f'\nMAE  = mean(|Error|)  = {np.mean(np.abs(errors)):.2f}')
print(f'MSE  = mean(Error²)   = {np.mean(errors**2):.2f}')
print(f'RMSE = √MSE           = {np.sqrt(np.mean(errors**2)):.2f}')
print(f'\n👉 Notice Store D: error of 80 contributes 80 to MAE sum but 6400 to MSE sum!')
print(f'   MSE punishes that big error {6400/80:.0f}× more than MAE does.')

In [ ]:
# --- Example 2: Loss curves — MAE (V-shape) vs MSE (parabola) ---
error_range = np.linspace(-100, 100, 500)
mae_loss = np.abs(error_range)
mse_loss = error_range ** 2

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: MAE loss curve
axes[0].plot(error_range, mae_loss, color='steelblue', linewidth=2)
axes[0].set_xlabel('Error (actual − predicted)', fontsize=11)
axes[0].set_ylabel('Loss', fontsize=11)
axes[0].set_title('MAE Loss: |error|\n(V-shape — sharp corner at 0)', fontsize=12)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[0].annotate('Kink: slope jumps from −1 to +1\ninstantly — no gradual transition\nerror=0.001 → gradient=1 (full speed!)\nerror=1000  → gradient=1 (same speed!)',
                 xy=(0, 0), xytext=(12, 55),
                 arrowprops=dict(arrowstyle='->', color='red'),
                 fontsize=9, color='red')
# Show the constant gradient on left and right
axes[0].annotate('slope = −1 everywhere\n(same step size far or near)', xy=(-60, 60), fontsize=9, color='purple', ha='center')
axes[0].annotate('slope = +1 everywhere\n(same step size far or near)', xy=(60, 60), fontsize=9, color='purple', ha='center')
axes[0].grid(True, alpha=0.3)

# Plot 2: MSE loss curve
axes[1].plot(error_range, mse_loss, color='coral', linewidth=2)
axes[1].set_xlabel('Error (actual − predicted)', fontsize=11)
axes[1].set_ylabel('Loss', fontsize=11)
axes[1].set_title('MSE Loss: error²\n(Smooth curve — gradient shrinks near minimum)', fontsize=12)
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[1].annotate('Gradient = 2×error (proportional!)\nerror=0.001 → gradient=0.002 (crawling)\nerror=50    → gradient=100 (decent)\nerror=1000  → gradient=2000 (flying)\n→ built-in speed controller',
                 xy=(0, 0), xytext=(15, 5500),
                 arrowprops=dict(arrowstyle='->', color='green'),
                 fontsize=9, color='green')
# Show gradient magnitude at different points
axes[1].annotate('gradient=200\n(big step — far away)', xy=(-100, 10000), fontsize=8, color='purple', ha='center')
axes[1].annotate('gradient=0.02\n(tiny step — almost there)', xy=(-10, 100), fontsize=8, color='purple', ha='center')
axes[1].grid(True, alpha=0.3)

# Plot 3: Overlay — how they diverge for large errors
axes[2].plot(error_range, mae_loss, color='steelblue', linewidth=2, label='MAE (linear)')
axes[2].plot(error_range, mse_loss / 100, color='coral', linewidth=2, label='MSE/100 (quadratic)')
axes[2].set_xlabel('Error (actual − predicted)', fontsize=11)
axes[2].set_ylabel('Loss (MSE scaled down for comparison)', fontsize=11)
axes[2].set_title('MAE vs MSE: How They Diverge\n(MSE explodes for large errors)', fontsize=12)
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Key visual takeaways:')
print('  Both MAE and MSE have gradient = 0 at the minimum. Both grow as error increases.')
print('  The difference is HOW they grow:')
print()
print('  • MAE: gradient jumps to ±1 instantly and STAYS at 1 no matter how far you are')
print('    error=0.001 → gradient=1 | error=1000 → gradient=1 (same speed!)')
print('    → No speed control. Takes the same size step whether close or far from the answer.')
print()
print('  • MSE: gradient = 2×error — PROPORTIONAL to how far off you are')
print('    error=0.001 → gradient=0.002 | error=1000 → gradient=2000')
print('    → Built-in speed controller. Big steps when far, tiny steps when close.')
print()
print('  Analogy: parking a car')
print('    MAE = driving at constant speed, slamming brakes at the spot (overshoot risk)')
print('    MSE = gradually slowing down as you approach, gliding in smoothly')

In [ ]:
# --- Example 3: Gradient Descent with MAE vs MSE loss on pizza data ---
# Simple 1D example: predict sales from employees
np.random.seed(42)
x_demo = np.array([3, 5, 7, 4, 6, 8, 2, 9, 5, 7], dtype=float)
y_demo = np.array([350, 500, 620, 410, 540, 700, 280, 750, 480, 610], dtype=float)

# Gradient descent with MSE loss
def gd_mse(x, y, lr=0.005, n_iters=200):
    w, b = 0.0, 0.0
    losses = []
    for _ in range(n_iters):
        pred = w * x + b
        error = y - pred
        loss = np.mean(error**2)
        losses.append(loss)
        w += lr * 2 * np.mean(error * x)
        b += lr * 2 * np.mean(error)
    return losses, w, b

# Gradient descent with MAE loss
def gd_mae(x, y, lr=0.5, n_iters=200):
    w, b = 0.0, 0.0
    losses = []
    for _ in range(n_iters):
        pred = w * x + b
        error = y - pred
        loss = np.mean(np.abs(error))
        losses.append(loss)
        # Subgradient: sign of error
        grad_sign = np.sign(error)
        w += lr * np.mean(grad_sign * x)
        b += lr * np.mean(grad_sign)
    return losses, w, b

mse_losses, w_mse, b_mse = gd_mse(x_demo, y_demo)
mae_losses, w_mae, b_mae = gd_mae(x_demo, y_demo)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Convergence comparison
axes[0].plot(mse_losses, color='coral', linewidth=2, label='MSE Loss')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('MSE: Smooth Convergence')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(mae_losses, color='steelblue', linewidth=2, label='MAE Loss')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('MAE Loss')
axes[1].set_title('MAE: Jagged Convergence (kink causes oscillation)')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.suptitle('Gradient Descent Convergence: MSE vs MAE', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'MSE model: sales = {w_mse:.1f} × employees + {b_mse:.1f}')
print(f'MAE model: sales = {w_mae:.1f} × employees + {b_mae:.1f}')
print(f'\n👉 MSE converges smoothly to a stable minimum.')
print(f'   MAE oscillates near the minimum because the gradient is always ±1 (never shrinks).')

In [ ]:
# --- Example 4: Outlier sensitivity — MAE vs MSE ---
# Same data but with one outlier store
x_clean = np.array([3, 5, 7, 4, 6, 8, 2, 9, 5, 7], dtype=float)
y_clean = np.array([350, 500, 620, 410, 540, 700, 280, 750, 480, 610], dtype=float)
y_outlier = y_clean.copy()
y_outlier[7] = 2000  # Store 8 has a data entry error: $2000 instead of $750

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x_line = np.linspace(1, 10, 100)

for ax, y_data, title in zip(axes, [y_clean, y_outlier],
    ['Clean Data', 'With Outlier (Store 8: $2000)']):
    ax.scatter(x_clean, y_data, color='steelblue', s=80, zorder=5)

    # Fit with MSE (OLS)
    from numpy.polynomial.polynomial import polyfit
    b_m, w_m = polyfit(x_clean, y_data, 1)
    ax.plot(x_line, b_m + w_m * x_line, color='coral', linewidth=2,
            label=f'MSE line (slope={w_m:.1f})')

    # Fit with MAE (median regression via simple approach)
    from scipy.optimize import minimize
    def mae_obj(params):
        return np.mean(np.abs(y_data - (params[0] * x_clean + params[1])))
    result = minimize(mae_obj, [50, 200], method='Nelder-Mead')
    w_a, b_a = result.x
    ax.plot(x_line, w_a * x_line + b_a, color='steelblue', linewidth=2,
            linestyle='--', label=f'MAE line (slope={w_a:.1f})')

    ax.set_xlabel('Employees')
    ax.set_ylabel('Daily Sales ($)')
    ax.set_title(title, fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Outlier Impact: MSE vs MAE Regression Lines', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Key insight:')
print('  • Clean data: both lines are nearly identical')
print('  • With outlier: MSE line gets pulled toward the outlier (error² = huge)')
print('  • MAE line barely moves — it\'s robust to outliers')
print('  • This is the main reason to choose MAE over MSE in practice')

### 🤔 What is R² and why do we need it?

MSE and RMSE tell you the **size** of the errors, but they don't tell you if that's good or bad. Is an RMSE of $15 good? It depends — $15 is great if sales range from $100 to $1000, but terrible if sales range from $10 to $20.

**R² (R-squared)** solves this by measuring the **proportion of variance explained** by the model. It answers: "How much better is my model than just predicting the average every time?"

---

**The intuition:**

Imagine the dumbest possible model — it ignores all features and always predicts the **mean** of y (e.g., "every store makes $405"). This "dumb model" has some total error. R² measures how much of that error your model eliminates.

```
Dumb model (predict mean every time):
  S1: actual=$520, predict=$405, error² = (520−405)² = 13,225
  S2: actual=$310, predict=$405, error² = (310−405)² =  9,025
  S3: actual=$620, predict=$405, error² = (620−405)² = 46,225
  SS_tot = 13,225 + 9,025 + 46,225 + ... = total squared error of the dumb model

Your model (uses features):
  S1: actual=$520, predict=$529, error² = (520−529)² =     81
  S2: actual=$310, predict=$330, error² = (310−330)² =    400
  S3: actual=$620, predict=$628, error² = (620−628)² =     64
  SS_res = 81 + 400 + 64 + ... = total squared error of YOUR model
```

---

**The formula:**

```
R² = 1 − (SS_res / SS_tot)

Where:
  SS_tot = Σ(yᵢ − ȳ)²    ← total variance (how far each point is from the mean)
  SS_res = Σ(yᵢ − ŷᵢ)²   ← residual variance (how far each point is from your prediction)
```

**How to read it:**

```
R² = 0.0  → Your model is no better than predicting the mean (useless)
R² = 0.5  → Your model explains 50% of the variance (decent)
R² = 0.9  → Your model explains 90% of the variance (great)
R² = 1.0  → Your model explains 100% of the variance (perfect — suspicious!)
R² < 0    → Your model is WORSE than predicting the mean (something is very wrong)
```

**Visual way to think about it:**

```
Total Variance (SS_tot)
┌─────────────────────────────────────────────────┐
│                                                 │
│   Explained by model (SS_reg)     Unexplained   │
│   ████████████████████████████    ░░ (SS_res)   │
│                                                 │
└─────────────────────────────────────────────────┘

R² = Explained / Total = 1 − Unexplained / Total
```

---

**Why Adjusted R²?**

R² has a flaw: it **always increases** when you add more features, even useless ones. Adding "store_id" as a feature would increase R², but the model would be worse on new data.

**Adjusted R²** penalizes for extra features:

```
Adjusted R² = 1 − [(1 − R²)(n − 1) / (n − p − 1)]

Where:
  n = number of data points
  p = number of features
```

If a new feature doesn't improve the model enough to justify its complexity, Adjusted R² will **drop**. Rule: if Adjusted R² goes down when you add a feature, remove that feature.

In [ ]:
# Compute all metrics step by step
residuals = y - y_pred_simple
n = len(y)
p = 1  # one feature

mse = np.mean(residuals**2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(residuals))
ss_res = np.sum(residuals**2)
ss_tot = np.sum((y - y_mean)**2)
r2 = 1 - ss_res / ss_tot
adj_r2 = 1 - ((1 - r2) * (n - 1)) / (n - p - 1)

print('EVALUATION METRICS')
print('=' * 50)
print(f'MSE  = (1/n)Σ(yᵢ − ŷᵢ)²  = {mse:.2f}')
print(f'RMSE = √MSE               = ${rmse:.2f}')
print(f'MAE  = (1/n)Σ|yᵢ − ŷᵢ|    = ${mae:.2f}')
print(f'\nSS_res = Σ(yᵢ − ŷᵢ)²    = {ss_res:.2f}')
print(f'SS_tot = Σ(yᵢ − ȳ)²       = {ss_tot:.2f}')
print(f'R²     = 1 − SS_res/SS_tot = {r2:.4f}')
print(f'Adj R² = 1 − [(1−R²)(n−1)/(n−p−1)] = {adj_r2:.4f}')
print(f'\n📊 Employees alone explains {r2*100:.1f}% of sales variance')
print(f'   Predictions are off by ~${rmse:.0f} on average')

---
## Part 3: Multiple Linear Regression

### 🤔 Why use multiple features instead of just one?

Simple regression uses one feature. But daily sales depend on **many things** — 
employees, rating, AND ad spend. Multiple regression uses all of them simultaneously.

**The key insight:** Each coefficient now represents that feature's **unique contribution**, 
holding all other features constant. This is called **ceteris paribus** ("all else equal").

You might notice the Employees coefficient changes from simple to multiple regression. 
That's because in the simple model, Employees was a **proxy** for everything — 
stores with more employees also tend to have higher ratings and bigger ad budgets. 
Multiple regression separates these effects.

In [ ]:
# Multiple regression with all features
features = ['employees', 'rating', 'ad_spend']
X = df[features]
y = df['daily_sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model_multi = LinearRegression()
model_multi.fit(X_train, y_train)
y_pred_multi = model_multi.predict(X_test)

print('MULTIPLE LINEAR REGRESSION')
print('=' * 50)
print(f'ŷ = {model_multi.intercept_:.2f}', end='')
for feat, coef in zip(features, model_multi.coef_):
    sign = '+' if coef >= 0 else ''
    print(f' {sign}{coef:.2f}×{feat}', end='')
print()

print(f'\nCoefficient Interpretation (holding others constant):')
for feat, coef in zip(features, model_multi.coef_):
    print(f'  {feat}: each +1 → ${coef:.2f} daily sales')

r2_multi = r2_score(y_test, y_pred_multi)
rmse_multi = np.sqrt(mean_squared_error(y_test, y_pred_multi))
print(f'\nR² = {r2_multi:.4f} (vs simple: {r2:.4f})')
print(f'RMSE = ${rmse_multi:.2f} (vs simple: ${rmse:.2f})')
print(f'\n🎯 Multiple regression is much better — uses all the information!')

### 3.1 Actual vs Predicted Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Simple model
y_pred_test_simple = model_simple.predict(X_test[['employees']])
axes[0].scatter(y_test, y_pred_test_simple, alpha=0.5, color='steelblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0].set_xlabel('Actual Sales ($)')
axes[0].set_ylabel('Predicted Sales ($)')
axes[0].set_title(f'Simple Model (R²={r2_score(y_test, y_pred_test_simple):.3f})')

# Multiple model
axes[1].scatter(y_test, y_pred_multi, alpha=0.5, color='coral')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[1].set_xlabel('Actual Sales ($)')
axes[1].set_ylabel('Predicted Sales ($)')
axes[1].set_title(f'Multiple Model (R²={r2_multi:.3f})')

plt.suptitle('Actual vs Predicted: Closer to red line = better', fontsize=12)
plt.tight_layout()
plt.show()

---
## Part 4: Normal Equation vs Gradient Descent

### 🤔 What is the Normal Equation?

The Normal Equation is a **closed-form solution** — you plug in the data and get the exact 
optimal weights in one computation:

**β = (XᵀX)⁻¹Xᵀy**

This works by setting the derivative of the loss function to zero and solving algebraically. 
It's like finding the bottom of a bowl by calculus instead of rolling a ball downhill.

**Pros:** Exact answer, no hyperparameters, one computation.
**Cons:** Requires matrix inversion (O(p³) — slow for many features), 
fails if features are perfectly correlated (singular matrix).

In [ ]:
# Normal Equation: β = (XᵀX)⁻¹Xᵀy
X_mat = np.column_stack([np.ones(len(X_train)), X_train.values])
y_vec = y_train.values

beta_normal = np.linalg.inv(X_mat.T @ X_mat) @ X_mat.T @ y_vec

print('NORMAL EQUATION: β = (XᵀX)⁻¹Xᵀy')
print('=' * 50)
print(f'β₀ (intercept):  {beta_normal[0]:.4f}')
for i, feat in enumerate(features):
    print(f'β{i+1} ({feat:>10}): {beta_normal[i+1]:.4f}')

print(f'\nsklearn intercept: {model_multi.intercept_:.4f}')
print(f'sklearn coefs:     {model_multi.coef_}')
print('\n✅ Normal equation gives the same answer as sklearn!')

### 🤔 What is Gradient Descent and why do we need it?

Gradient descent is an **iterative** algorithm — it starts with random weights and 
improves them step by step:

1. Compute predictions with current weights
2. Compute the **gradient** (direction of steepest increase in error)
3. Move weights in the **opposite direction** (downhill)
4. Repeat until the error stops decreasing

**Why not always use the Normal Equation?** For large datasets (millions of rows, 
thousands of features), the matrix inversion is too slow. Gradient descent scales much better.

**Why scale features first?** Without scaling, features with large values (ad_spend: 50–300) 
dominate the gradient, causing zig-zagging and slow convergence. 
StandardScaler puts all features on the same scale (mean=0, std=1).

In [ ]:
# Gradient Descent from scratch
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Add intercept column
X_gd = np.column_stack([np.ones(len(X_train_scaled)), X_train_scaled])
n_samples, n_features = X_gd.shape

# Initialize
beta = np.zeros(n_features)
lr = 0.01
n_iters = 1000
loss_history = []

for i in range(n_iters):
    y_hat = X_gd @ beta
    residuals = y_train.values - y_hat
    mse_loss = np.mean(residuals**2)
    loss_history.append(mse_loss)
    gradient = -(2/n_samples) * (X_gd.T @ residuals)
    beta = beta - lr * gradient

print('GRADIENT DESCENT FROM SCRATCH')
print('=' * 50)
print(f'Learning rate: {lr}, Iterations: {n_iters}')
print(f'Initial MSE: {loss_history[0]:.2f}')
print(f'Final MSE:   {loss_history[-1]:.2f}')
print(f'\nConverged weights (scaled features):')
print(f'  β₀={beta[0]:.2f}, β₁={beta[1]:.2f}, β₂={beta[2]:.2f}, β₃={beta[3]:.2f}')

# Plot convergence
plt.figure(figsize=(10, 4))
plt.plot(loss_history, color='steelblue')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.title('Gradient Descent Convergence')
plt.grid(True, alpha=0.3)
plt.show()

---
## Part 5: Assumptions & Diagnostics

### 🤔 Why check assumptions?

Linear regression makes assumptions about the data. If these are violated, 
the model's predictions and confidence intervals may be **unreliable**.

The 4 diagnostic plots below check:
1. **Residuals vs Predicted**: Should be random scatter (no pattern = linearity ✅, uniform band = homoscedasticity ✅)
2. **Q-Q Plot**: Points should follow the diagonal line (= normally distributed errors ✅)
3. **Histogram of Residuals**: Should be bell-shaped (= normal distribution ✅)
4. **Correlation Heatmap**: No feature pair should have correlation > 0.8 (= no multicollinearity ✅)

If you see a curved pattern in the residual plot, your model is missing a non-linear relationship. 
If the residuals fan out (cone shape), the variance isn't constant — try log-transforming y.

In [ ]:
residuals_multi = y_test.values - y_pred_multi

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Residuals vs Predicted (Linearity + Homoscedasticity)
axes[0,0].scatter(y_pred_multi, residuals_multi, alpha=0.5, color='steelblue')
axes[0,0].axhline(y=0, color='r', linestyle='--')
axes[0,0].set_xlabel('Predicted')
axes[0,0].set_ylabel('Residuals')
axes[0,0].set_title('1. Residuals vs Predicted\n(Check: random scatter = good)')

# 2. Q-Q Plot (Normality)
from scipy import stats
sorted_res = np.sort(residuals_multi)
theoretical = stats.norm.ppf(np.linspace(0.01, 0.99, len(sorted_res)))
axes[0,1].scatter(theoretical, sorted_res, alpha=0.5, color='coral')
axes[0,1].plot([-3, 3], [-3*np.std(residuals_multi), 3*np.std(residuals_multi)], 'r--')
axes[0,1].set_xlabel('Theoretical Quantiles')
axes[0,1].set_ylabel('Sample Quantiles')
axes[0,1].set_title('2. Q-Q Plot\n(Check: points on line = normal)')

# 3. Histogram of Residuals
axes[1,0].hist(residuals_multi, bins=20, color='steelblue', edgecolor='white', alpha=0.7)
axes[1,0].set_xlabel('Residual')
axes[1,0].set_ylabel('Frequency')
axes[1,0].set_title('3. Residual Distribution\n(Check: bell-shaped = normal)')

# 4. Feature Correlation Heatmap (Multicollinearity)
corr = df[features].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=axes[1,1],
            fmt='.2f', square=True)
axes[1,1].set_title('4. Feature Correlations\n(Check: no high correlations)')

plt.suptitle('Linear Regression Diagnostic Plots', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Assumption Checks:')
print(f'  Residual mean: {np.mean(residuals_multi):.2f} (should be ~0) ✅')
print(f'  Residual std:  {np.std(residuals_multi):.2f}')
shapiro_stat, shapiro_p = stats.shapiro(residuals_multi)
print(f'  Shapiro-Wilk normality p-value: {shapiro_p:.4f}', '✅' if shapiro_p > 0.05 else '⚠️')

### 🤔 What is Multicollinearity and why is it a problem?

Multicollinearity = two or more features are **highly correlated** with each other.

**Pizza example:** Suppose we add a feature `total_staff = employees + managers`. Now `total_staff` and `employees` move almost identically — if one goes up, the other goes up.

**Why it's bad:**

The model needs to figure out: "how much does each feature contribute to sales?" But if `employees` and `total_staff` carry the same information, the model can't tell which one deserves the credit. It might give employees β=+100 and total_staff β=−75, or employees β=+25 and total_staff β=0 — both fit the data equally well.

```
Without multicollinearity:          With multicollinearity:
  employees β = 25 (stable)           employees β = +100 (unstable!)
  rating    β = 50 (stable)           total_staff β = −75 (unstable!)
  ad_spend  β = 1.5 (stable)          rating β = 50 (still stable)

The correlated features' coefficients become UNRELIABLE.
Small changes in data → wildly different β values.
```

**What breaks:**
- Coefficients become unstable (change a lot with small data changes)
- Standard errors blow up → p-values become meaningless
- You can't interpret individual coefficients anymore
- Predictions may still be fine! (the problem is interpretation, not prediction)

**How to detect — VIF (Variance Inflation Factor):**
- VIF = 1: no correlation with other features
- VIF = 5: moderately correlated (warning)
- VIF > 10: highly correlated (problem!)

**How to fix:**
- Remove one of the correlated features
- Use Ridge regression (L2 penalty stabilizes correlated coefficients)
- Use PCA to combine correlated features into uncorrelated components

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MULTICOLLINEARITY DEMO — see it break with correlated features
# ═══════════════════════════════════════════════════════════════

from statsmodels.stats.outliers_influence import variance_inflation_factor

# Original features — should be fine
print('ORIGINAL FEATURES — VIF check')
print('=' * 50)
for i, feat in enumerate(['employees', 'rating', 'ad_spend']):
    vif = variance_inflation_factor(X_train.values, i)
    status = '✅' if vif < 5 else '⚠️' if vif < 10 else '❌'
    print(f'  {feat:<12} VIF = {vif:.2f} {status}')

# Now add a correlated feature: total_staff ≈ employees + noise
X_train_bad = X_train.copy()
X_train_bad['total_staff'] = X_train['employees'] + np.random.randint(1, 3, len(X_train))

print(f'\nAFTER ADDING total_staff (≈ employees + small noise)')
print('=' * 50)
for i, feat in enumerate(['employees', 'rating', 'ad_spend', 'total_staff']):
    vif = variance_inflation_factor(X_train_bad.values, i)
    status = '✅' if vif < 5 else '⚠️' if vif < 10 else '❌'
    print(f'  {feat:<12} VIF = {vif:.2f} {status}')

# Show how coefficients become unstable
print(f'\nCOEFFICIENT INSTABILITY:')
print('-' * 50)
# Fit on original
from sklearn.linear_model import LinearRegression
m1 = LinearRegression().fit(X_train, y_train)
print(f'  Original:  employees={m1.coef_[0]:.2f}, rating={m1.coef_[1]:.2f}, ad_spend={m1.coef_[2]:.2f}')

# Fit with correlated feature
m2 = LinearRegression().fit(X_train_bad, y_train)
print(f'  + total_staff: employees={m2.coef_[0]:.2f}, rating={m2.coef_[1]:.2f}, ad_spend={m2.coef_[2]:.2f}, total_staff={m2.coef_[3]:.2f}')

# Fit again with slightly different data
X_train_bad2 = X_train.copy()
X_train_bad2['total_staff'] = X_train['employees'] + np.random.randint(1, 3, len(X_train))
m3 = LinearRegression().fit(X_train_bad2, y_train)
print(f'  + total_staff (different noise): employees={m3.coef_[0]:.2f}, rating={m3.coef_[1]:.2f}, ad_spend={m3.coef_[2]:.2f}, total_staff={m3.coef_[3]:.2f}')

print(f'\n👉 employees and total_staff coefficients swing wildly!')
print(f'   But rating and ad_spend stay stable — they are not correlated.')
print(f'   The PREDICTIONS are still similar — it is the INTERPRETATION that breaks.')

### 🤔 What is Autocorrelation and when does it matter?

Autocorrelation = errors are **correlated with each other** across observations.

**When does this happen?** Mostly with **time-series data** — data ordered by time.

**Pizza example:** Suppose you predict daily sales for one store over 30 days. If Monday's prediction is too high (positive error), Tuesday's prediction is likely also too high — because the same underlying factor (maybe a road closure) affects both days.

```
Day    Actual   Predicted   Error
Mon     $400      $500      -100  ← overpredicted
Tue     $420      $510       -90  ← also overpredicted (correlated!)
Wed     $410      $505       -95  ← still overpredicted
Thu     $600      $490      +110  ← now underpredicted
Fri     $620      $500      +120  ← also underpredicted (correlated!)

Errors follow a PATTERN — they don't bounce randomly.
This violates the independence assumption.
```

**Why it's bad:**

Linear regression assumes each error is **independent** — like flipping a coin each time. The math behind standard errors, confidence intervals, and p-values all rely on this assumption.

When errors are autocorrelated, the model thinks it has **more independent information than it actually does.**

**Analogy:** Imagine you survey 100 people about a restaurant. If all 100 are strangers, you have 100 independent opinions — strong evidence. But if 80 of them are from the same family who all had the same bad experience, you really only have ~20 independent opinions. The survey THINKS it has 100 data points, but the real information content is much less.

That's exactly what happens with autocorrelated errors:

```
What the model thinks:       What's actually happening:
  50 independent errors        50 errors, but groups of 5-10
  → lots of information          are nearly identical (correlated)
  → small standard errors      → much less real information
  → tight confidence bands     → standard errors SHOULD be bigger
  → small p-values             → confidence bands SHOULD be wider
  → "this feature matters!"    → "maybe this feature doesn't matter"
```

**The result:** You conclude a feature is important (small p-value) when it might not be. Your confidence intervals look precise but they're falsely narrow. The predictions themselves may still be OK — it's the **statistical conclusions** ("is this feature significant?") that become unreliable.

**How to detect — Durbin-Watson (DW) test:**
- DW ≈ 2.0: no autocorrelation (good)
- DW close to 0: positive autocorrelation (errors follow each other)
- DW close to 4: negative autocorrelation (errors alternate)

**When NOT to worry:**
- Cross-sectional data (our 200 pizza stores are independent locations)
- Each row is a different entity, not the same entity over time
- Autocorrelation is mainly a time-series concern

**How to fix (when it matters):**
- Add lagged features (yesterday's sales as a predictor)
- Use time-series models (ARIMA, Prophet) instead of plain linear regression
- Add time-based features (day of week, month, trend)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# AUTOCORRELATION DEMO — independent vs correlated errors
# ═══════════════════════════════════════════════════════════════

from scipy.stats import pearsonr

np.random.seed(42)

# Case 1: Independent errors (cross-sectional — our pizza stores)
errors_independent = np.random.normal(0, 30, 50)

# Case 2: Autocorrelated errors (time-series — daily sales)
errors_correlated = [np.random.normal(0, 30)]
for i in range(49):
    # Each error is 80% of the previous + some noise
    errors_correlated.append(0.8 * errors_correlated[-1] + np.random.normal(0, 15))
errors_correlated = np.array(errors_correlated)

# Durbin-Watson statistic (manual calculation)
def durbin_watson(residuals):
    diff = np.diff(residuals)
    return np.sum(diff**2) / np.sum(residuals**2)

dw_indep = durbin_watson(errors_independent)
dw_corr = durbin_watson(errors_correlated)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(errors_independent, 'o-', color='steelblue', markersize=4, alpha=0.7)
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[0].set_title(f'Independent Errors (cross-sectional)\nDW = {dw_indep:.2f} (close to 2 = good)')
axes[0].set_xlabel('Store index')
axes[0].set_ylabel('Residual')
axes[0].grid(True, alpha=0.3)

axes[1].plot(errors_correlated, 'o-', color='coral', markersize=4, alpha=0.7)
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].set_title(f'Autocorrelated Errors (time-series)\nDW = {dw_corr:.2f} (close to 0 = bad)')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Residual')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Autocorrelation: Independent vs Correlated Errors', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Durbin-Watson statistic:')
print(f'  Independent errors: DW = {dw_indep:.2f} (close to 2 = no autocorrelation ✅)')
print(f'  Correlated errors:  DW = {dw_corr:.2f} (close to 0 = positive autocorrelation ❌)')
print(f'\n👉 Left plot: errors bounce randomly — each store is independent')
print(f'   Right plot: errors follow a pattern — today\'s error predicts tomorrow\'s')
print(f'   Our pizza store data is cross-sectional, so autocorrelation is not a concern here.')

---
## Part 6: Regularization — Ridge, Lasso, Elastic Net

### 🤔 What is Regularization and why do we need it?

With many features, a model can **overfit** — it memorizes the training data (including noise) 
and performs poorly on new data. Regularization adds a **penalty** to the loss function 
that discourages large coefficients:

- **Ridge (L2)**: Penalty = λΣβⱼ² → shrinks all coefficients toward zero, but never to exactly zero
- **Lasso (L1)**: Penalty = λΣ|βⱼ| → can shrink coefficients to **exactly zero** (feature selection!)
- **Elastic Net**: Combines both L1 and L2

**λ (alpha)** controls the penalty strength:
- λ = 0: No penalty (standard OLS)
- λ = large: Heavy penalty (coefficients near zero, underfitting)

Use **cross-validation** to find the best λ.

### 🤔 Wait — what are βⱼ and yᵢ? Are they from the same row?

No! They're different things:

- **yᵢ** = the actual sales for store **i** (one per row in your data)
- **βⱼ** = the learned coefficient for feature **j** (one per column — shared across ALL rows)

Think of it this way:

```
Your data (each row is a store i):
┌───────┬───────────┬────────┬──────────┬─────────────┐
│ Store │ employees │ rating │ ad_spend │ daily_sales  │
│  (i)  │   (x₁ᵢ)  │ (x₂ᵢ) │  (x₃ᵢ)  │    (yᵢ)     │
├───────┼───────────┼────────┼──────────┼─────────────┤
│  i=1  │     6     │  4.2   │   150    │    $520     │
│  i=2  │     3     │  3.1   │    80    │    $310     │
│  i=3  │     8     │  4.8   │   250    │    $680     │
│  ...  │    ...    │  ...   │   ...    │    ...      │
└───────┴───────────┴────────┴──────────┴─────────────┘
                                          ↑ changes per row

The model learns ONE set of β values (shared for all rows):
  β₀ = 30    (intercept)
  β₁ = 25    (employees coefficient)
  β₂ = 50    (rating coefficient)
  β₃ = 1.5   (ad_spend coefficient)
       ↑ same for every store

Prediction for store i=1:
  ŷ₁ = β₀ + β₁×x₁₁ + β₂×x₂₁ + β₃×x₃₁
     = 30 + 25×6   + 50×4.2 + 1.5×150
     = 30 + 150    + 210    + 225
     = $615

Prediction for store i=2:
  ŷ₂ = 30 + 25×3   + 50×3.1 + 1.5×80
     = 30 + 75     + 155    + 120
     = $380

Same β values, different x values per row → different predictions.
```

**In the loss function:**

```
MSE = (1/n) Σᵢ (yᵢ − ŷᵢ)²     ← sums over rows (stores)
Penalty = λ Σⱼ βⱼ²              ← sums over columns (features)

Ridge Loss = MSE + Penalty
           = (1/n) Σᵢ (yᵢ − ŷᵢ)² + λ(β₁² + β₂² + β₃²)
```

The subscript **i** loops over rows (stores). The subscript **j** loops over features (columns). They're independent.

In [70]:
# ═══════════════════════════════════════════════════════════════
# SEE IT WITH REAL DATA: βⱼ vs yᵢ on actual pizza stores
# ═══════════════════════════════════════════════════════════════

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

# Use the first 5 stores from our training data as a concrete example
sample = X_train.head(5).copy()
sample['actual_sales (yᵢ)'] = y_train.head(5).values

print('OUR REAL DATA — first 5 stores')
print('Each ROW is a store (i). Each COLUMN is a feature (j).')
print('=' * 65)
print(sample.to_string(index=True))

# Fit OLS on full training data
ols = LinearRegression().fit(X_train, y_train)

print(f'\nLEARNED COEFFICIENTS (same for ALL stores):')
print(f'  β₀ (intercept)  = {ols.intercept_:.2f}')
for j, feat in enumerate(['employees', 'rating', 'ad_spend']):
    print(f'  β{j+1} ({feat:>10}) = {ols.coef_[j]:.2f}')

print(f'\nPREDICTION FOR EACH STORE (same β, different x per row):')
print('-' * 65)
for idx, row in X_train.head(5).iterrows():
    pred = ols.intercept_
    parts = [f'{ols.intercept_:.1f}']
    for j, feat in enumerate(['employees', 'rating', 'ad_spend']):
        val = row[feat]
        coef = ols.coef_[j]
        pred += coef * val
        parts.append(f'{coef:.1f}×{val}')
    actual = y_train.loc[idx]
    print(f'  Store {idx}: ŷ = {" + ".join(parts)} = ${pred:.0f}  (actual: ${actual})')

OUR REAL DATA — first 5 stores
Each ROW is a store (i). Each COLUMN is a feature (j).
     employees  rating  ad_spend  actual_sales (yᵢ)
169          3     2.8       238                626
97           8     3.0       101                498
31           4     3.0       154                531
12           9     2.7        78                480
35           6     4.6       294                855

LEARNED COEFFICIENTS (same for ALL stores):
  β₀ (intercept)  = 20.03
  β1 ( employees) = 24.53
  β2 (    rating) = 51.01
  β3 (  ad_spend) = 1.55

PREDICTION FOR EACH STORE (same β, different x per row):
-----------------------------------------------------------------
  Store 169: ŷ = 20.0 + 24.5×3.0 + 51.0×2.8 + 1.5×238.0 = $605  (actual: $626)
  Store 97: ŷ = 20.0 + 24.5×8.0 + 51.0×3.0 + 1.5×101.0 = $526  (actual: $498)
  Store 31: ŷ = 20.0 + 24.5×4.0 + 51.0×3.0 + 1.5×154.0 = $510  (actual: $531)
  Store 12: ŷ = 20.0 + 24.5×9.0 + 51.0×2.7 + 1.5×78.0 = $499  (actual: $480)
  Store 35: ŷ = 20

In [71]:
# ═══════════════════════════════════════════════════════════════
# NOW SEE HOW REGULARIZATION CHANGES THE β VALUES
# Same stores, same x values — only β changes!
# ═══════════════════════════════════════════════════════════════

# Scale features (required for fair regularization)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)

# Fit all three models
ols_sc = LinearRegression().fit(X_train_sc, y_train)
ridge_sc = Ridge(alpha=10).fit(X_train_sc, y_train)
lasso_sc = Lasso(alpha=10).fit(X_train_sc, y_train)

print('HOW REGULARIZATION CHANGES β (on scaled features)')
print('=' * 65)
print(f'{"":>12} {"β₁ employees":>14} {"β₂ rating":>14} {"β₃ ad_spend":>14}')
print('-' * 65)
print(f'{"OLS":>12} {ols_sc.coef_[0]:>14.2f} {ols_sc.coef_[1]:>14.2f} {ols_sc.coef_[2]:>14.2f}')
print(f'{"Ridge α=10":>12} {ridge_sc.coef_[0]:>14.2f} {ridge_sc.coef_[1]:>14.2f} {ridge_sc.coef_[2]:>14.2f}')
print(f'{"Lasso α=10":>12} {lasso_sc.coef_[0]:>14.2f} {lasso_sc.coef_[1]:>14.2f} {lasso_sc.coef_[2]:>14.2f}')

# Show the penalty each model pays
print(f'\nPENALTY EACH MODEL PAYS:')
print(f'  OLS:   no penalty (free to use any β)')
print(f'  Ridge: λ × Σβⱼ² = 10 × ({ridge_sc.coef_[0]:.2f}² + {ridge_sc.coef_[1]:.2f}² + {ridge_sc.coef_[2]:.2f}²) = {10 * np.sum(ridge_sc.coef_**2):.2f}')
lasso_penalty = 10 * np.sum(np.abs(lasso_sc.coef_))
print(f'  Lasso: λ × Σ|βⱼ| = 10 × (|{lasso_sc.coef_[0]:.2f}| + |{lasso_sc.coef_[1]:.2f}| + |{lasso_sc.coef_[2]:.2f}|) = {lasso_penalty:.2f}')

# Show predictions for first 3 stores under each model
print(f'\nPREDICTIONS — same stores, different β:')
print(f'{"Store":>8} {"Actual":>8} {"OLS":>8} {"Ridge":>8} {"Lasso":>8}')
print('-' * 45)
X_sample_sc = scaler.transform(X_train.head(5))
for i in range(5):
    actual = y_train.iloc[i]
    p_ols = ols_sc.predict(X_sample_sc[i:i+1])[0]
    p_ridge = ridge_sc.predict(X_sample_sc[i:i+1])[0]
    p_lasso = lasso_sc.predict(X_sample_sc[i:i+1])[0]
    print(f'{i+1:>8} {actual:>8} {p_ols:>8.0f} {p_ridge:>8.0f} {p_lasso:>8.0f}')

print(f'\n👉 Key takeaway:')
print(f'   yᵢ changes per row (each store has different sales)')
print(f'   βⱼ is shared across ALL rows (one coefficient per feature)')
print(f'   Regularization shrinks the β values — the data stays the same')

HOW REGULARIZATION CHANGES β (on scaled features)
               β₁ employees      β₂ rating    β₃ ad_spend
-----------------------------------------------------------------
         OLS          69.53          36.68         107.58
  Ridge α=10          65.37          34.70         100.78
  Lasso α=10          60.35          27.42          98.74

PENALTY EACH MODEL PAYS:
  OLS:   no penalty (free to use any β)
  Ridge: λ × Σβⱼ² = 10 × (65.37² + 34.70² + 100.78²) = 156333.41
  Lasso: λ × Σ|βⱼ| = 10 × (|60.35| + |27.42| + |98.74|) = 1865.06

PREDICTIONS — same stores, different β:
   Store   Actual      OLS    Ridge    Lasso
---------------------------------------------
       1      626      605      608      622
       2      498      526      534      541
       3      531      510      518      531
       4      480      499      509      518
       5      855      858      845      834

👉 Key takeaway:
   yᵢ changes per row (each store has different sales)
   βⱼ is shared across ALL

### 🤔 How are regularized coefficients actually calculated?

Let's trace through the math for each method.

---

**OLS (no penalty) — the baseline:**

```
Loss = (1/n) Σ(yᵢ − ŷᵢ)²

Closed-form: β = (XᵀX)⁻¹ Xᵀy
```

The model is free to make coefficients as large as it wants.

---

**Ridge (L2) — squared penalty:**

```
Loss = (1/n) Σ(yᵢ − ŷᵢ)² + λ × Σβⱼ²
                ↑ fit the data    ↑ keep coefficients small
```

If β₁ = 50, the penalty adds 50² = 2500 to the loss. The model must balance:
- "Fit the data well" (minimize MSE)
- "Keep coefficients small" (minimize penalty)

**Ridge closed-form:**
```
β_ridge = (XᵀX + λI)⁻¹ Xᵀy
                ↑ this is the ONLY difference from OLS!
```

Adding λI to XᵀX does two things:
1. Makes the matrix always invertible (even with correlated features)
2. Shrinks all coefficients toward zero — larger λ = more shrinkage

---

**Lasso (L1) — absolute value penalty:**

```
Loss = (1/n) Σ(yᵢ − ŷᵢ)² + λ × Σ|βⱼ|
```

Why can Lasso zero out coefficients but Ridge can't?

```
Ridge penalty gradient: d/dβ (β²) = 2β
  → When β is small (say 0.01), gradient = 0.02 (tiny push toward 0)
  → The push gets weaker as β approaches 0 — it never quite gets there

Lasso penalty gradient: d/dβ (|β|) = sign(β) = ±1
  → When β is small (say 0.01), gradient = 1 (FULL push toward 0)
  → The push is constant regardless of how small β is
  → If the feature's contribution to MSE reduction < λ, it gets zeroed out
```

No closed-form solution — must use coordinate descent or subgradient methods.

---

**Elastic Net (L1 + L2):**

```
Loss = (1/n) Σ(yᵢ − ŷᵢ)² + λ₁ × Σ|βⱼ| + λ₂ × Σβⱼ²
                                  ↑ selection    ↑ stability
```

Gets Lasso's ability to zero out features + Ridge's stability with correlated features.

---

Let's verify this by computing Ridge coefficients **by hand** using the closed-form formula and comparing with sklearn.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# RIDGE COEFFICIENTS BY HAND: β = (XᵀX + λI)⁻¹ Xᵀy
# ═══════════════════════════════════════════════════════════════

# Use scaled training data (same as sklearn uses internally)
X_mat = X_train_scaled  # already scaled from earlier
y_vec = y_train.values
lam = 1.0  # λ = 1.0

n_samples, n_feats = X_mat.shape

# Step 1: XᵀX
print(X_mat.T.shape)
print(X_mat.shape)
XtX = X_mat.T @ X_mat
print(XtX)
print('Step 1: XᵀX (feature cross-products)')
print(f'  Shape: {XtX.shape}')
print(f'  Diagonal (feature variances × n): {np.diag(XtX).round(2)}')

# Step 2: Add λI (the regularization!)
lambda_I = lam * np.eye(n_feats)
print(f'\nStep 2: λI (penalty matrix, λ={lam})')
print(f'  This adds {lam} to each diagonal element of XᵀX')
print(f'  Before: diag = {np.diag(XtX).round(2)}')
print(f'  After:  diag = {np.diag(XtX + lambda_I).round(2)}')

# Step 3: Invert (XᵀX + λI)
XtX_reg_inv = np.linalg.inv(XtX + lambda_I)

# Step 4: Multiply by Xᵀy
Xty = X_mat.T @ y_vec
beta_ridge_hand = XtX_reg_inv @ Xty

# Compare with sklearn
ridge_sklearn = Ridge(alpha=lam).fit(X_mat, y_vec)

print(f'\nStep 3-4: β = (XᵀX + λI)⁻¹ Xᵀy')
print(f'\n{"Feature":<12} {"Hand":>10} {"sklearn":>10} {"Match?":>8}')
print('-' * 42)
features_list = ['employees', 'rating', 'ad_spend']
for feat, bh, bs in zip(features_list, beta_ridge_hand, ridge_sklearn.coef_):
    match = '✅' if abs(bh - bs) < 0.01 else '❌'
    print(f'{feat:<12} {bh:>10.4f} {bs:>10.4f} {match:>8}')

print(f'\n✅ Hand calculation matches sklearn!')

(3, 140)
(140, 3)
Step 1: XᵀX (feature cross-products)
  Shape: (3, 3)
  Diagonal (feature variances × n): [140. 140. 140.]

Step 2: λI (penalty matrix, λ=1.0)
  This adds 1.0 to each diagonal element of XᵀX
  Before: diag = [140. 140. 140.]
  After:  diag = [141. 141. 141.]

Step 3-4: β = (XᵀX + λI)⁻¹ Xᵀy

Feature            Hand    sklearn   Match?
------------------------------------------
employees       69.0903    69.0903        ✅
rating          36.4737    36.4737        ✅
ad_spend       106.8586   106.8586        ✅

✅ Hand calculation matches sklearn!


In [77]:
# ═══════════════════════════════════════════════════════════════
# VISUALIZE: How λ changes the coefficients
# ═══════════════════════════════════════════════════════════════

lambdas = [0, 0.1, 1, 10, 100, 1000]
print('HOW λ SHRINKS RIDGE COEFFICIENTS')
print('=' * 75)
print(f'{"λ":<8}', end='')
for feat in features_list:
    print(f'{feat:>14}', end='')
print(f'{"Σβ²":>14}')
print('-' * 75)

for lam in lambdas:
    if lam == 0:
        # OLS: β = (XᵀX)⁻¹ Xᵀy
        beta = np.linalg.inv(X_mat.T @ X_mat) @ (X_mat.T @ y_vec)
    else:
        # Ridge: β = (XᵀX + λI)⁻¹ Xᵀy
        beta = np.linalg.inv(X_mat.T @ X_mat + lam * np.eye(n_feats)) @ (X_mat.T @ y_vec)
    penalty = np.sum(beta**2)
    print(f'{lam:<8.1f}', end='')
    for b in beta:
        print(f'{b:>14.4f}', end='')
    print(f'{penalty:>14.2f}')

print(f'\n👉 As λ increases:')
print(f'   • All coefficients shrink toward 0 (but never reach it)')
print(f'   • Σβ² (total coefficient magnitude) decreases')
print(f'   • The model becomes simpler but may underfit')
print(f'\n👉 λ=0 is just OLS. λ=∞ would make all coefficients = 0.')

HOW λ SHRINKS RIDGE COEFFICIENTS
λ            employees        rating      ad_spend           Σβ²
---------------------------------------------------------------------------
0.0            69.5299       36.6818      107.5801      17753.44
0.1            69.4857       36.6609      107.5075      17730.15
1.0            69.0903       36.4737      106.8586      17522.57
10.0           65.3693       34.6977      100.7783      15633.34
100.0          42.4595       23.2416       64.2784       6474.69
1000.0          9.4107        5.3490       13.9344        311.34

👉 As λ increases:
   • All coefficients shrink toward 0 (but never reach it)
   • Σβ² (total coefficient magnitude) decreases
   • The model becomes simpler but may underfit

👉 λ=0 is just OLS. λ=∞ would make all coefficients = 0.


In [74]:
# ═══════════════════════════════════════════════════════════════
# LASSO: Why it zeros out coefficients (coordinate descent demo)
# ═══════════════════════════════════════════════════════════════

from sklearn.linear_model import Lasso

print('LASSO: WATCH COEFFICIENTS HIT ZERO')
print('=' * 75)
print(f'{"λ":<8}', end='')
for feat in features_list:
    print(f'{feat:>14}', end='')
print(f'{"# nonzero":>14}')
print('-' * 75)

for lam in [0.01, 0.1, 1, 5, 10, 50, 100]:
    lasso = Lasso(alpha=lam, max_iter=10000).fit(X_mat, y_vec)
    n_nonzero = np.sum(lasso.coef_ != 0)
    print(f'{lam:<8.2f}', end='')
    for b in lasso.coef_:
        marker = ' ← ZERO!' if b == 0 else ''
        print(f'{b:>14.4f}', end='')
    print(f'{n_nonzero:>14}')

print(f'\n👉 Lasso vs Ridge difference:')
print(f'   Ridge penalty gradient: d/dβ(β²) = 2β → push weakens as β→0 (never reaches 0)')
print(f'   Lasso penalty gradient: d/dβ(|β|) = ±1 → constant push regardless of β size')
print(f'   If a feature\'s MSE contribution < λ, Lasso zeros it out completely')

LASSO: WATCH COEFFICIENTS HIT ZERO
λ            employees        rating      ad_spend     # nonzero
---------------------------------------------------------------------------
0.01           69.5207       36.6726      107.5712             3
0.10           69.4381       36.5892      107.4917             3
1.00           68.6122       35.7552      106.6959             3
5.00           64.9414       32.0487      103.1590             3
10.00          60.3529       27.4155       98.7380             3
50.00          23.4483        0.0000       62.8243             2
100.00          0.0000        0.0000       14.4125             1

👉 Lasso vs Ridge difference:
   Ridge penalty gradient: d/dβ(β²) = 2β → push weakens as β→0 (never reaches 0)
   Lasso penalty gradient: d/dβ(|β|) = ±1 → constant push regardless of β size
   If a feature's MSE contribution < λ, Lasso zeros it out completely


In [75]:
# Compare OLS vs Ridge vs Lasso vs Elastic Net
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]

print('REGULARIZATION COMPARISON')
print('=' * 70)
print(f'{"Model":<25} {"RMSE":>8} {"R²":>8}  Coefficients')
print('-' * 70)

# OLS baseline
ols = LinearRegression().fit(X_train_scaled, y_train)
ols_pred = ols.predict(X_test_scaled)
print(f'{"OLS (no penalty)":<25} {np.sqrt(mean_squared_error(y_test, ols_pred)):>8.2f} {r2_score(y_test, ols_pred):>8.4f}  {np.round(ols.coef_, 2)}')

# Ridge (L2)
for alpha in [0.1, 1.0, 10.0]:
    ridge = Ridge(alpha=alpha).fit(X_train_scaled, y_train)
    ridge_pred = ridge.predict(X_test_scaled)
    print(f'{f"Ridge (α={alpha})":<25} {np.sqrt(mean_squared_error(y_test, ridge_pred)):>8.2f} {r2_score(y_test, ridge_pred):>8.4f}  {np.round(ridge.coef_, 2)}')

# Lasso (L1)
for alpha in [0.1, 1.0, 10.0]:
    lasso = Lasso(alpha=alpha).fit(X_train_scaled, y_train)
    lasso_pred = lasso.predict(X_test_scaled)
    print(f'{f"Lasso (α={alpha})":<25} {np.sqrt(mean_squared_error(y_test, lasso_pred)):>8.2f} {r2_score(y_test, lasso_pred):>8.4f}  {np.round(lasso.coef_, 2)}')

# Elastic Net
enet = ElasticNet(alpha=1.0, l1_ratio=0.5).fit(X_train_scaled, y_train)
enet_pred = enet.predict(X_test_scaled)
print(f'{"Elastic Net (α=1.0)":<25} {np.sqrt(mean_squared_error(y_test, enet_pred)):>8.2f} {r2_score(y_test, enet_pred):>8.4f}  {np.round(enet.coef_, 2)}')

print('\nKey observations:')
print('  • Ridge shrinks coefficients but never zeros them')
print('  • Lasso can zero out coefficients (feature selection!)')
print('  • High α → more shrinkage → simpler model')

REGULARIZATION COMPARISON
Model                         RMSE       R²  Coefficients
----------------------------------------------------------------------
OLS (no penalty)             32.86   0.9442  [ 69.53  36.68 107.58]
Ridge (α=0.1)                32.84   0.9443  [ 69.49  36.66 107.51]
Ridge (α=1.0)                32.72   0.9447  [ 69.09  36.47 106.86]
Ridge (α=10.0)               32.53   0.9454  [ 65.37  34.7  100.78]
Lasso (α=0.1)                32.83   0.9444  [ 69.44  36.59 107.49]
Lasso (α=1.0)                32.56   0.9453  [ 68.61  35.76 106.7 ]
Lasso (α=10.0)               34.09   0.9400  [60.35 27.42 98.74]
Elastic Net (α=1.0)          50.81   0.8667  [47.77 25.81 72.79]

Key observations:
  • Ridge shrinks coefficients but never zeros them
  • Lasso can zero out coefficients (feature selection!)
  • High α → more shrinkage → simpler model


### 6.1 Regularization Path — How Coefficients Shrink

In [ ]:
alphas_range = np.logspace(-2, 3, 50)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ridge path
ridge_coefs = []
for a in alphas_range:
    ridge = Ridge(alpha=a).fit(X_train_scaled, y_train)
    ridge_coefs.append(ridge.coef_)
ridge_coefs = np.array(ridge_coefs)

for i, feat in enumerate(features):
    axes[0].plot(alphas_range, ridge_coefs[:, i], label=feat)
axes[0].set_xscale('log')
axes[0].set_xlabel('α (regularization strength)')
axes[0].set_ylabel('Coefficient value')
axes[0].set_title('Ridge (L2): Coefficients shrink but never reach 0')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Lasso path
lasso_coefs = []
for a in alphas_range:
    lasso = Lasso(alpha=a, max_iter=10000).fit(X_train_scaled, y_train)
    lasso_coefs.append(lasso.coef_)
lasso_coefs = np.array(lasso_coefs)

for i, feat in enumerate(features):
    axes[1].plot(alphas_range, lasso_coefs[:, i], label=feat)
axes[1].set_xscale('log')
axes[1].set_xlabel('α (regularization strength)')
axes[1].set_ylabel('Coefficient value')
axes[1].set_title('Lasso (L1): Coefficients can hit exactly 0')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Regularization Paths', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 7: Polynomial Regression — When Linear Isn't Enough

### 🤔 What if the relationship isn't linear?

Linear regression fits a straight line. But what if the true relationship is curved? 
**Polynomial regression** adds powers of x (x², x³, ...) as new features:

- Degree 1: ŷ = β₀ + β₁x (straight line — may underfit)
- Degree 2: ŷ = β₀ + β₁x + β₂x² (parabola — often just right)
- Degree 15: ŷ = β₀ + β₁x + ... + β₁₅x¹⁵ (wiggly — overfits!)

This is a perfect demonstration of the **bias-variance tradeoff**:
- Too simple → misses the pattern (high bias)
- Too complex → fits the noise (high variance)
- Just right → captures the true pattern

In [ ]:
# Demonstrate underfitting vs overfitting with polynomial degree
np.random.seed(42)
x_poly = np.sort(np.random.uniform(0, 10, 30))
y_poly = 3 + 2*x_poly - 0.3*x_poly**2 + np.random.normal(0, 2, 30)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x_plot = np.linspace(0, 10, 100)

for ax, degree, title in zip(axes, [1, 2, 15],
    ['Degree 1: Underfitting', 'Degree 2: Just Right', 'Degree 15: Overfitting']):
    poly = PolynomialFeatures(degree)
    X_p = poly.fit_transform(x_poly.reshape(-1, 1))
    model_p = LinearRegression().fit(X_p, y_poly)
    X_plot = poly.transform(x_plot.reshape(-1, 1))
    y_plot = model_p.predict(X_plot)

    train_r2 = model_p.score(X_p, y_poly)
    ax.scatter(x_poly, y_poly, alpha=0.6, color='steelblue')
    ax.plot(x_plot, y_plot, 'r-', linewidth=2)
    ax.set_title(f'{title}\nTrain R²={train_r2:.3f}')
    ax.set_ylim(y_poly.min()-5, y_poly.max()+5)
    ax.grid(True, alpha=0.3)

plt.suptitle('Bias-Variance Tradeoff with Polynomial Degree', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Key takeaway:')
print('  Degree 1:  High bias (misses the curve)')
print('  Degree 2:  Just right (captures the true pattern)')
print('  Degree 15: High variance (fits noise, will fail on new data)')

---
## Part 8: Cross-Validation

### 🤔 What is Cross-Validation and why is it better than a single train/test split?

A single train/test split is **random** — you might get lucky or unlucky with which data 
ends up in each set. **K-fold cross-validation** gives a more reliable estimate:

1. Split data into K equal parts (folds)
2. Train on K-1 folds, test on the remaining fold
3. Repeat K times (each fold gets to be the test set once)
4. Average the K scores

This uses **all data for both training and testing** (just not at the same time). 
The standard deviation of the K scores tells you how **stable** the model is — 
low std = consistent performance, high std = sensitive to which data it sees.

In [ ]:
# 5-fold cross-validation
models = {
    'OLS': LinearRegression(),
    'Ridge (α=1)': Ridge(alpha=1.0),
    'Lasso (α=1)': Lasso(alpha=1.0),
    'Elastic Net': ElasticNet(alpha=1.0, l1_ratio=0.5),
}

print('5-FOLD CROSS-VALIDATION')
print('=' * 50)
print(f'{"Model":<20} {"Mean R²":>10} {"Std":>10}')
print('-' * 50)

for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
    print(f'{name:<20} {scores.mean():>10.4f} {scores.std():>10.4f}')

print('\nCross-validation gives a more reliable estimate of model performance')
print('than a single train/test split.')

---
## Summary

You've learned:
- **Simple Linear Regression**: β₁ = Cov(x,y)/Var(x), computed by hand
- **Multiple Linear Regression**: Isolating each feature's unique effect
- **Normal Equation vs Gradient Descent**: Two ways to find optimal weights
- **Evaluation**: MSE, RMSE, MAE, R², Adjusted R²
- **Assumptions**: Linearity, normality, homoscedasticity, independence
- **Regularization**: Ridge (L2), Lasso (L1), Elastic Net
- **Polynomial Regression**: When linear isn't enough
- **Cross-Validation**: Reliable model evaluation